# Generación de Streamgraph de Anomalías Climáticas (IA)

Este notebook procesa los datos y genera un **Streamgraph** (gráfico de corrientes) que evidencia la asimetría y el ensanchamiento de la volatilidad climática. Cumple con la regla estricta de **no utilizar** gráficos comunes como barras, líneas, torta o dispersión.

Pasos ejecutados:
1. Carga de datos limpios (`GlobalLandTemperaturesByCountry_cleaned.csv`).
2. Mapeo de países a continentes.
3. Cálculo de anomalías térmicas base.
4. Visualización usando `stackplot` con `baseline='wiggle'`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime

# 1. Cargar el archivo
df = pd.read_csv('../data/GlobalLandTemperaturesByCountry_cleaned.csv')

# 2. Filtrar hasta el año 2013 y obtener década
df['dt'] = pd.to_datetime(df['dt'])
df = df[df['dt'].dt.year <= 2013].copy()
df['Year'] = df['dt'].dt.year
df['Decade'] = (df['Year'] // 10) * 10

# 3. Agrupar países por continente o grandes regiones
# Mapeo manual para los países más representativos del dataset
continent_map = {
    'Africa': ['Algeria', 'Angola', 'Benin', 'Botswana', 'Burkina Faso', 'Burundi', 'Cameroon', 'Cape Verde', 'Central African Republic', 'Chad', 'Comoros', 'Congo', 'Djibouti', 'Egypt', 'Equatorial Guinea', 'Eritrea', 'Ethiopia', 'Gabon', 'Gambia', 'Ghana', 'Guinea', 'Guinea Bissau', 'Ivory Coast', 'Kenya', 'Lesotho', 'Liberia', 'Libya', 'Madagascar', 'Malawi', 'Mali', 'Mauritania', 'Mauritius', 'Morocco', 'Mozambique', 'Namibia', 'Niger', 'Nigeria', 'Rwanda', 'Sao Tome And Principe', 'Senegal', 'Seychelles', 'Sierra Leone', 'Somalia', 'South Africa', 'Sudan', 'Swaziland', 'Tanzania', 'Togo', 'Tunisia', 'Uganda', 'Zambia', 'Zimbabwe', 'Democratic Republic Of The Congo'],
    'Asia': ['Afghanistan', 'Bahrain', 'Bangladesh', 'Bhutan', 'Brunei', 'Burma', 'Cambodia', 'China', 'India', 'Indonesia', 'Iran', 'Iraq', 'Israel', 'Japan', 'Jordan', 'Kazakhstan', 'Kuwait', 'Kyrgyzstan', 'Laos', 'Lebanon', 'Malaysia', 'Maldives', 'Mongolia', 'Nepal', 'North Korea', 'Oman', 'Pakistan', 'Philippines', 'Qatar', 'Saudi Arabia', 'Singapore', 'South Korea', 'Sri Lanka', 'Syria', 'Taiwan', 'Tajikistan', 'Thailand', 'Timor Leste', 'Turkey', 'Turkmenistan', 'United Arab Emirates', 'Uzbekistan', 'Vietnam', 'Yemen'],
    'Europe': ['Albania', 'Andorra', 'Austria', 'Belarus', 'Belgium', 'Bosnia And Herzegovina', 'Bulgaria', 'Croatia', 'Cyprus', 'Czech Republic', 'Denmark', 'Estonia', 'Finland', 'France', 'Germany', 'Greece', 'Hungary', 'Iceland', 'Ireland', 'Italy', 'Latvia', 'Liechtenstein', 'Lithuania', 'Luxembourg', 'Macedonia', 'Malta', 'Moldova', 'Monaco', 'Montenegro', 'Netherlands', 'Norway', 'Poland', 'Portugal', 'Romania', 'Russia', 'San Marino', 'Serbia', 'Slovakia', 'Slovenia', 'Spain', 'Sweden', 'Switzerland', 'Ukraine', 'United Kingdom'],
    'North America': ['Antigua And Barbuda', 'Bahamas', 'Barbados', 'Belize', 'Canada', 'Costa Rica', 'Cuba', 'Dominica', 'Dominican Republic', 'El Salvador', 'Greenland', 'Grenada', 'Guatemala', 'Haiti', 'Honduras', 'Jamaica', 'Mexico', 'Nicaragua', 'Panama', 'Saint Kitts And Nevis', 'Saint Lucia', 'Saint Vincent And The Grenadines', 'Trinidad And Tobago', 'United States'],
    'South America': ['Argentina', 'Bolivia', 'Brazil', 'Chile', 'Colombia', 'Ecuador', 'Guyana', 'Paraguay', 'Peru', 'Suriname', 'Uruguay', 'Venezuela'],
    'Oceania': ['Australia', 'Fiji', 'Kiribati', 'Marshall Islands', 'Micronesia', 'Nauru', 'New Zealand', 'Palau', 'Papua New Guinea', 'Samoa', 'Solomon Islands', 'Tonga', 'Tuvalu', 'Vanuatu']
}

country_to_continent = {country: cont for cont, countries in continent_map.items() for country in countries}
df['Continent'] = df['Country'].map(country_to_continent)
df = df.dropna(subset=['Continent']) # Omitir países que no se mapearon para claridad en el gráfico de áreas grandes

# 4. Calcular anomalías por continente respecto a un período base (1951-1980)
base_period = df[(df['Year'] >= 1951) & (df['Year'] <= 1980)]
baseline = base_period.groupby('Continent')['AverageTemperature'].mean().reset_index()
baseline = baseline.rename(columns={'AverageTemperature': 'BaseTemp'})

df = df.merge(baseline, on='Continent')
df['Anomaly'] = df['AverageTemperature'] - df['BaseTemp']

# 5. Agrupar anomalías promedio por década
decade_continent = df.groupby(['Decade', 'Continent'])['Anomaly'].mean().reset_index()
pivot_df = decade_continent.pivot(index='Decade', columns='Continent', values='Anomaly').fillna(0)

# Para visualizar la volatilidad como el grosor en el streamgraph, tomamos el valor absoluto
# ya que las variaciones extremas (tanto frías como cálidas) representan el ensanchamiento
pivot_df_abs = pivot_df.abs()

# Filtrar a partir de 1850 para tener datos consistentes
pivot_df_abs = pivot_df_abs[pivot_df_abs.index >= 1850]

x = pivot_df_abs.index.values
y = [pivot_df_abs[col].values for col in pivot_df_abs.columns]

# 6. Generar el Streamgraph
plt.style.use('dark_background')
fig, ax = plt.subplots(figsize=(14, 7))

# Colores contrastantes para los continentes
colors = ['#ff6b6b', '#4ecdc4', '#45b7d1', '#f9ca24', '#6c5ce7', '#ff9ff3']

# Graficar usando baseline='wiggle' (Streamgraph)
ax.stackplot(x, y, labels=pivot_df_abs.columns, baseline='wiggle', colors=colors, alpha=0.85)

# Títulos y estilos
ax.set_title('Asimetría del Cambio Climático: Evolución de la Volatilidad Térmica Global (1850-2010)', fontsize=18, pad=25, color='white', fontweight='bold')
ax.set_xlabel('Década', fontsize=14, color='lightgray', labelpad=10)
ax.set_ylabel('Magnitud de Anomalía (Volatilidad)', fontsize=14, color='lightgray', labelpad=10)
ax.tick_params(colors='lightgray', labelsize=12)

# Quitar bordes para estética limpia
for spine in ax.spines.values():
    spine.set_visible(False)

# Cuadrícula muy sutil
ax.grid(axis='x', color='gray', linestyle='--', alpha=0.2)

# Leyenda elegante
plt.legend(loc='upper left', bbox_to_anchor=(0.02, 0.98), frameon=False, labelcolor='white', fontsize=12, ncol=3)

# 7. Guardar y mostrar
plt.tight_layout()
plt.savefig('grafico_ia.png', transparent=False, facecolor=fig.get_facecolor(), dpi=300)
plt.show()
print("Gráfico generado con éxito en grafico_ia.png")
